# 04 — Querying the confidence strata

The call relation is stored as three **pure relations** —
`calls_resolved` / `calls_assumed` / `calls_unresolved` — and that stratification
(D3 in `docs/VISION.md`) is what makes the facts usable for *verification*
rather than just exploration. It hands us the standard sound-analysis lattice:

* **must** (`ir.calls_must` = resolved only) — the under-approximation. A
  violation found here is a **definite** finding.
* **may** (`ir.calls` = all three strata) — the over-approximation. A violation
  found *only* here is **possible**, and the non-resolved edges on its witness
  are exactly the unknowns a future SMT layer would reason over.

This notebook uses the strata as a query substrate over the MOM6 + FMS2 corpus:
stratum censuses, classifying the unresolved frontier, must-vs-may reachability
with witnessing paths, and the "certain island" — a taste of the invariant
style the planned query layer (Phase 5) will formalize.

Prerequisites: notebook 01 for concepts; corpus access as in notebook 02.

## Parameters

In [ ]:
import os
from pathlib import Path

CORPUS = Path(os.environ.get(
    "GROUNDLINE_CORPUS",
    "/glade/work/altuntas/turbo-stack/bin/flang_ptree/MOM6_using_FMS2",
))
assert CORPUS.is_dir(), (
    f"{CORPUS} not found — set GROUNDLINE_CORPUS to a directory of *_ptree dumps."
)
paths = sorted(p for p in CORPUS.rglob("*_ptree") if p.is_file())
print(f"{len(paths)} parse-tree dumps under {CORPUS}")

## Build the forest (~40 s) and take the census

In [ ]:
from groundline.parse_forest import ParseForest

forest = ParseForest(paths)
ir = forest.ir

n_may = len(ir.calls)
for name in ("calls_resolved", "calls_assumed", "calls_unresolved"):
    rel = getattr(ir, name)
    print(f"{name:18} {len(rel):6}  ({len(rel) / n_may:5.1%} of may)")
print(f"{'may / must':18} {n_may:6} / {len(ir.calls_must)}")

## The `assumed` stratum: genuine dynamic dispatch

After Phase 2, `assumed` is *only* produced by dynamic dispatch — a type-bound
call on a polymorphic receiver, where the frontend records the **declared**
type's binding while an override may win at runtime. Everything a compiler can
settle statically is `resolved`, which is why this stratum is so small.

In [ ]:
from collections import Counter

assumed_callers = sorted({caller for caller, _ in ir.calls_assumed})
print(f"{len(ir.calls_assumed)} assumed edges from {len(assumed_callers)} routines\n")
for caller, callee in sorted(ir.calls_assumed)[:8]:
    print(f"  {caller}  -?->  {callee}")

## The `unresolved` frontier, classified

Unresolved *targets* are first-class entities (`defined=False`), so we can ask
who they are. Two distinct populations:

* **module-pinned** atoms (`netcdf::nf90_open`) — the use-chain or sema's name
  mangling pins the owning module even though its source was never parsed:
  external libraries, precisely identified.
* **bare-name** atoms — nothing pins a module. This mixes genuine externals
  (`mpi_allreduce`) with a known frontend residue: intrinsics missing from its
  hardcoded list (`sqrt`, `exp`, `loc`, …) surface here instead of being
  filtered (recorded in `docs/DEVLOG.md`, Phase 2 — visible, not hidden).

In [ ]:
refs = Counter(callee for _, callee in ir.calls_unresolved)
pinned = {t for t in refs if ir.get(t) and ir.get(t).scope}
bare = set(refs) - pinned
print(f"{len(refs)} distinct unresolved targets: "
      f"{len(pinned)} module-pinned, {len(bare)} bare\n")

print("most-referenced module-pinned targets:")
for t, n in refs.most_common():
    if t in pinned:
        print(f"  {n:4}  {t}")

print("\nmost-referenced bare targets:")
for t, n in [(t, n) for t, n in refs.most_common() if t in bare][:12]:
    print(f"  {n:4}  {t}")

## must vs may on the call graph

`get_call_graph()` labels every edge with its stratum; `must_only=True` keeps
the compiler-certain subgraph (same node set — only edges are filtered). The
difference between the two graphs *is* the uncertainty frontier.

(The graph's edge counts sit slightly below the raw relation counts: the graph's
nodes are subroutines and functions, so the few call events recorded from
`program` units or module-level code are relation facts without a graph edge.)

In [ ]:
g_may = forest.get_call_graph()
g_must = forest.get_call_graph(must_only=True)
print(f"may:  {g_may.number_of_edges()} edges")
print(f"must: {g_must.number_of_edges()} edges")
print(f"frontier: {g_may.number_of_edges() - g_must.number_of_edges()} edges "
      "a proof could not rely on")

## The certain island

Which routines have a **fully compiler-certain transitive call closure** — no
assumed or unresolved edge reachable from them at all? For those, must- and
may-analyses agree, so any invariant checked on their subgraph is settled
without touching the unknowns. (This is the relational skeleton of VISION D6's
"provable in isolation" criterion.)

One reverse reachability pass answers it: mark every routine with a
non-resolved out-edge as *tainted*, walk the call graph backwards from all of
them, and whatever is never reached is the island.

In [ ]:
from groundline.ir import RESOLVED

tainted = {
    u for u, v, d in g_may.edges(data=True) if d["confidence"] != RESOLVED
}
affected = set(tainted)
stack = list(tainted)
while stack:
    node = stack.pop()
    for pred in g_may.predecessors(node):
        if pred not in affected:
            affected.add(pred)
            stack.append(pred)

island = [n for n in g_may.nodes if n not in affected and n.defined]
defined_nodes = [n for n in g_may.nodes if n.defined]
print(f"{len(tainted)} routines with a non-resolved out-edge")
print(f"{len(affected)} can reach one (their proofs would carry assumptions)")
print(f"{len(island)} of {len(defined_nodes)} defined routines are fully certain "
      f"({len(island) / len(defined_nodes):.0%})")

## Reachability under uncertainty: a witnessed counterexample

The invariant style groundline is heading toward: *"routine X never reaches
routine Y."* Evaluate it on both views —

* holds on **may** → provably safe, regardless of how the unknowns resolve;
* fails on **must** → definitely violated, with the call chain as the witness;
* holds on must but fails on may → **conditional**: the violation exists only
  if the guessed edges are real. The witnessing chain's non-resolved links are
  the exact unknowns to discharge (or hand to an SMT layer: *does some / every
  resolution of these edges violate the invariant?* — VISION §3, Q4).

Below we pick, automatically, the dynamic-dispatch caller whose
may-reachability exceeds its must-reachability the most, and exhibit one
witnessed conditional path.

In [ ]:
import networkx as nx

by_id = {n.id: n for n in g_may.nodes}
candidates = sorted((by_id[c] for c in {c for c, _ in ir.calls_assumed} if c in by_id),
                    key=lambda n: n.id)
gap_of = {
    x: len(nx.descendants(g_may, x)) - len(nx.descendants(g_must, x))
    for x in candidates
}
x = max(candidates, key=lambda n: gap_of[n])
print(f"{x.id}: may-reaches {gap_of[x]} more routines than must-reaches\n")

must_reach = nx.descendants(g_must, x)
may_paths = nx.single_source_shortest_path(g_may, x)
conditional = [y for y in may_paths if y not in must_reach and y is not x]
# The most instructive witness: the longest (shortest) chain, so the resolved
# prefix and the uncertain link both show.
y = max(conditional, key=lambda y: (len(may_paths[y]), y.id))

print(f"invariant '{x.id} never reaches {y.id}':")
print("  on must-edges: HOLDS   on may-edges: FAILS — witness:")
path = may_paths[y]
for a, b in zip(path, path[1:]):
    conf = g_may.edges[a, b]["confidence"]
    marker = "   " if conf == RESOLVED else "!! "
    print(f"  {marker}{a.id}  -[{conf}]->  {b.id}")

Every `!!` link is an unknown standing between "possible" and "proven". Shrink
the assumed/unresolved strata (a better frontend, more parsed sources) and
conditional findings migrate to definite ones — without changing a single
query. That is the point of keeping confidence in the model.

## Where this goes

Phase 5 (see `docs/DESIGN.md` §4) turns these hand-written traversals into a
relational query layer — join, closure, difference over the same relations —
with a CLI so an invariant like the GPU-porting gate ("no new HostOnly edge
crosses into the ported set") can fail CI with the witnessing chain printed,
exactly like the one above.